In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from scipy.optimize import brute
# # Base directory
# base_dir = 'data/raw/'

# # Pattern to match all .txt files
# pattern = os.path.join(base_dir, '*.txt')

# # Find all files matching the pattern
# file_paths = glob.glob(pattern)

# # Sort the file paths
# file_paths.sort()

# # Read and concatenate
# all_dataframes = [pd.read_csv(file_path, sep=",", low_memory=False) for file_path in file_paths]
# combined_df = pd.concat(all_dataframes, ignore_index=True)

# # Clean column names if needed
# def clean_column_names(column_names):
#     return [col.replace('[', '').replace(']', '').strip() for col in column_names]

# combined_df.columns = clean_column_names(combined_df.columns)


combined_df = pd.read_csv('clean_data2023.csv')
# Verify the combined DataFrame
print(combined_df.columns.tolist())



In [ ]:
print(combined_df['DTE'].dtype)
print(combined_df['STRIKE'].dtype)
print(combined_df['UNDERLYING_LAST'].dtype)
print(combined_df['P_IV'].dtype)

# combined_df['DTE'] = pd.to_numeric(combined_df['DTE'], errors='coerce')
# combined_df['STRIKE_DISTANCE'] = pd.to_numeric(combined_df['STRIKE_DISTANCE'], errors='coerce')
# combined_df['UNDERLYING_LAST'] = pd.to_numeric(combined_df['UNDERLYING_LAST'], errors='coerce')
# combined_df['P_IV'] = pd.to_numeric(combined_df['P_IV'], errors='coerce')

# # Drop any rows where 'P_IV' is NaN since it's required for the loss function calculation
# combined_df = combined_df.dropna(subset=['P_IV'])
# combined_df = combined_df[combined_df['DTE'] != 0]



In [ ]:
combined_df = combined_df.dropna(subset=['P_IV'])
combined_df = combined_df.dropna(subset=['STRIKE'])
combined_df = combined_df.dropna(subset=['UNDERLYING_LAST'])
combined_df = combined_df.dropna(subset=['DTE'])
combined_df = combined_df[combined_df['DTE'] != 0]


In [ ]:
combined_df.columns.tolist()


In [ ]:


# Calculate Ft;T
# combined_df['Ft;T'] = combined_df['UNDERLYING_LAST'] * np.exp(combined_df['rf'] * (combined_df['DTE'] / 365))
combined_df['Ft;T'] = combined_df['UNDERLYING_LAST'] * 0.05 * (combined_df['DTE'] / 365)

# Ensure that 'STRIKE_DISTANCE' and 'Ft;T' are positive
combined_df = combined_df[(combined_df['STRIKE'] > 0) & (combined_df['Ft;T'] > 0)]

# Calculate the log-strike k = log(K/Ft;T)
combined_df['k'] = combined_df['Log-Strike']
# combined_df['k'] = np.log(combined_df['STRIKE']/combined_df['Ft;T'])

# Define kappa1 and kappa2 based on your model or estimation
kappa1 = 5.5
kappa2 = 0.1
combined_df.columns.tolist()
combined_df['k'].max()


In [ ]:
import numpy as np
from scipy.optimize import minimize, Bounds
from scipy.optimize import NonlinearConstraint
def ssvi_iv(params, k, T):
    rho, eta, gamma, V, V_prime, theta = params
    theta_T = theta * T + (V - theta) * (1 - np.exp(-kappa1 * T)) / kappa1 + (V_prime - theta) * (kappa1 / (kappa1 - kappa2)) * (((1 - np.exp(-kappa2 * T)) / kappa2) - (1 - np.exp(-kappa1 * T)) / kappa1)
    phi_theta_T = eta / (theta_T ** gamma)
    ssvi = (theta_T / 2) * (1 + rho * phi_theta_T * k + np.sqrt((phi_theta_T * k + rho) ** 2 + (1 - rho ** 2)))
    return np.sqrt(ssvi / T)

def ssvi_loss(params, k, T, observed_iv):
    predicted_iv = ssvi_iv(params, k, T)
    return np.sum((predicted_iv - observed_iv) ** 2)



In [ ]:
# def optimize_ssvi(combined_df):
#     k = combined_df['k']
#     T = combined_df['DTE'] / 365  # Convert DTE from days to years
#     observed_iv = combined_df['P_IV'] # Placeholder for the column with observed implied volatilities
#     #rho, eta, gamma, V, V_prime, theta
#     # Bounds for the parameters
#     bounds = Bounds([-1, 0, 0, 0, 0, 0], [1, 10, 1, 10, 10, 1])

#     # Multiple initial guesses
#     initial_guesses = [
#         [0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
#         [0.2, 0.2, 0.2, 0.2, 0.2, 0.2],
#         [-0.1, 0.3, 0.1, 0.1, 0.1, 0.1]
#     ]


    
#     best_result = None
#     for initial_params in initial_guesses:
#         # result = minimize(ssvi_loss, initial_params, args=(k, T, observed_iv), method='SLSQP', bounds=bounds)
#         result = minimize(ssvi_loss, initial_params, args=(k, T, observed_iv), method='TNC', bounds=bounds)
#         if best_result is None or result.fun < best_result.fun:
#             best_result = result

#     fitted_params = best_result.x
#     return fitted_params

# # Assuming combined_df is already defined with the necessary data
# fitted_params = optimize_ssvi(combined_df)
# print(f"Fitted SSVI Parameters: {fitted_params}")
# print("rho = ", fitted_params[0])
# print("eta = ", fitted_params[1])
# print("gamma = ", fitted_params[2])
# print("V = ", fitted_params[3])
# print("V prime = ", fitted_params[4])
# print("theta = ", fitted_params[5])


In [ ]:
%matplotlib widget
# Fitted SSVI Parameters
fitted_params = [-0.72669427,  1.1998262, 0.5, 1.355e-02,  3.302e-02,  1.100e-01] #min

# fitted_params = [-0.04994575,  0.82954736,  0.741216,    0.00234273,  0.02834748,  0.16998581] #dzh
# fitted_params = [-0.05905818,  1.94798225,  0.61510017,  0.00985712,  0.00784887,  0.09993211] #dzh2
# fitted_params = [-0.5,  1.19,  0.49,    0.04,  0.04,  0.11] #paper
# fitted_params = [-0.1840547,0.78967695, 0.69300519, 0.02351505, 0.04734758, 0.08732117] #cby
# fitted_params = [-0.18301757,  0.78737235,  0.66174031,  0.02090151,  0.05287529,0.0710549] #cby2



def ssvi_IV(params, k, T):
    rho, eta, gamma, V, V_prime, theta = params
    theta_T = theta * T + (V - theta) * (1 - np.exp(-kappa1 * T)) / kappa1 + (V_prime - theta) * (kappa1 / (kappa1 - kappa2)) * (((1 - np.exp(-kappa2 * T)) / kappa2) - ((1 - np.exp(-kappa1 * T)) / kappa1))
    theta_T = np.maximum(theta_T, 1e-10)
    phi_theta_T = eta / (theta_T ** gamma)
    ssvi = (theta_T / 2) * (1 + rho * phi_theta_T * k + np.sqrt((phi_theta_T * k + rho) ** 2 + (1 - rho ** 2)))
    return ssvi

def ssvi_surface(params, k_range, T_range):
    surface = np.zeros((len(k_range), len(T_range)))
    for i, strike in enumerate(k_range):
        for j, time_to_expiration in enumerate(T_range):
            value = np.sqrt(ssvi_IV(params, strike, time_to_expiration) / time_to_expiration)
            surface[i, j] = value
    return surface

# Define ranges for k and T
k_range = np.linspace(-2, 1, 50)  # strike range
T_range = np.linspace(0.1, 5, 50)      # time to expiration range

# Compute the SSVI surface
surface = ssvi_surface(fitted_params, k_range, T_range)

# Plotting the surface
K, T = np.meshgrid(k_range, T_range)
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(K, T, surface.T, cmap='viridis')
# ax.scatter(combined_df['k'], combined_df['DTE']/365, combined_df['P_IV'], color='r', s=3)

ax.set_xlabel('Log-moneyness')
ax.set_ylabel('Time to Expiration')
ax.set_zlabel('Implied Volatility')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt



def plot_ssvi_2d_multiple_T(params, k_range, T_values):
    plt.figure(figsize=(10, 6))
    for T in T_values:
        iv_values = [np.sqrt(ssvi_IV(params, k, T) / T) for k in k_range]
        plt.plot(k_range, iv_values, label=f'T = {T} years')
    plt.xlabel('Log-moneyness (K)')
    plt.ylabel('Implied Volatility (IV)')
    plt.title('2D SSVI Implied Volatility Plot for Different T')
    plt.legend()
    plt.grid(True)
    plt.show()

# Define range for k
k_range = np.linspace(-2, 2, 50)  # Log-moneyness range

# Fixed times to expiration to plot
T_values = [0.5, 1, 1.5, 2]

# Plot 2D SSVI IV curve for multiple T values
plot_ssvi_2d_multiple_T(fitted_params, k_range, T_values)


In [ ]:
def plot_iv_comparison(combined_df, fitted_params):
    k = combined_df['k']
    T = combined_df['DTE'] / 365  # Convert DTE from days to years
    observed_iv = combined_df['P_IV']

    # Calculate implied volatilities using the fitted parameters
    calculated_iv = [ssvi_iv(fitted_params, ki, Ti) for ki, Ti in zip(k, T)]

    # Plotting
    plt.figure(figsize=(10, 6))
    plt.plot(k, observed_iv, 'o', label='Market IV', color='blue', alpha = 0.1)
    plt.plot(k, calculated_iv, 'x',label='SSVI IV', color='red', alpha = 0.1)
    plt.xlabel('Strike Price (k)')
    plt.ylabel('Implied Volatility')
    plt.title('Market Implied Volatility vs. SSVI Implied Volatility')
    plt.legend()
    plt.grid(True)
    plt.show()

# Assuming combined_df is already defined and fitted_params is obtained
plot_iv_comparison(combined_df, fitted_params)


In [ ]:
def calculate_similarity(combined_df, fitted_params, threshold=0.05):
    k = combined_df['k']
    T = combined_df['DTE'] / 365  # Convert DTE from days to years
    observed_iv = combined_df['P_IV']

    # Calculate implied volatilities using the fitted parameters
    calculated_iv = np.array([ssvi_iv(fitted_params, ki, Ti) for ki, Ti in zip(k, T)])

    # Calculate the percentage difference
    percent_diff = np.abs((calculated_iv - observed_iv) / observed_iv)

    # Calculate the percentage of data points within the threshold
    similar_count = np.sum(percent_diff < threshold)
    total_count = len(observed_iv)
    similarity_percentage = (similar_count / total_count) * 100

    return similarity_percentage

threshold = 0.1
similarity_percentage = calculate_similarity(combined_df, fitted_params, threshold)
print(f"Percentage of data within {threshold*100}% similarity: {similarity_percentage}%")
